# TP4 -- ANOVA à deux facteurs et Tests Non Paramétriques

**Statistique Mathématique 3 -- L3 MIASHS**

---

## Introduction

Ce TP couvre :
1. **Simulations ANOVA** : comprendre le test de Fisher par la simulation
2. **ANOVA à deux facteurs** : effet simultané de deux facteurs
3. **Tests non paramétriques** : alternatives sans hypothèse de normalité

---
## 1. Simulations et ANOVA à 1 facteur

$F_{obs} = \frac{CMM}{CME}$ : rapport variance modèle / variance erreur. Plus l'erreur est faible, plus on rejette facilement.

In [ ]:
# Modèle sans erreur : 3 niveaux, moyennes 15, 20, 25
x = c(rep(15, 20), rep(20, 20), rep(25, 20))
a = c(rep("a", 20), rep("b", 20), rep("c", 20))
anova(lm(x ~ a))  # Avertissement : ajustement parfait

In [ ]:
# Ajout d'erreur avec sigma = 0.5
set.seed(42)
err = rnorm(60)
xe = x + 0.5 * err
anova(lm(xe ~ a))

In [ ]:
# Fonction automatisée
anovalue = function(x, a, sigma) {
  err = rnorm(length(x))
  xe = x + sigma * err
  return(anova(lm(xe ~ a))$Pr)
}

cat("sigma = 1  :", anovalue(x, a, 1), "\n")
cat("sigma = 5  :", anovalue(x, a, 5), "\n")
cat("sigma = 10 :", anovalue(x, a, 10), "\n")
cat("sigma = 15 :", anovalue(x, a, 15), "\n")
cat("sigma = 20 :", anovalue(x, a, 20), "\n")

La p-value fluctue à chaque appel (aléa). Près du seuil 5%, les résultats changent souvent → **attention au p-hacking** !

---
## 2. ANOVA à deux facteurs

### 2.1 Données de dureté de mèche

4 marques de mèches × 5 types de dureté d'acier. Réponse = vitesse de pénétration.

In [ ]:
options(OutDec = ",")
durtVit <- read.table("dureteMeche.txt", header = TRUE)
head(durtVit)
Y = durtVit$Y
D = factor(durtVit$D)
M = factor(durtVit$M)

In [ ]:
# Graphe d'interaction
interaction.plot(M, D, Y, main = "Graphe d'interaction",
                 xlab = "Mèche", ylab = "Moyenne de Y", col = 1:5, lwd = 2)

Les lignes ne se croisent pas → interaction **faible**.

In [ ]:
# ANOVA sans interaction (modèle additif)
summary(aov(Y ~ D + M))

In [ ]:
# ANOVA avec interaction
anova(lm(Y ~ D * M))

### 2.2 Pourquoi pas d'ANOVA à 1 facteur ici ?

In [ ]:
cat("=== Y ~ M ===\n")
summary(aov(Y ~ M))
cat("\n=== Y ~ D ===\n")
summary(aov(Y ~ D))

M semble significatif seul, mais pas D. En ANOVA à 2 facteurs, **les deux** sont significatifs. C'est leur **action jointe** qui compte.

### 2.3 ANOVA avec interaction (engrais)

In [ ]:
engrais = read.table("engraisRegion.txt", header = TRUE)
Y = engrais$Y
R = factor(engrais$R)
E = factor(engrais$E)
anova(lm(Y ~ R * E))

In [ ]:
anova(lm(Y ~ R + E))
interaction.plot(R, E, Y, main = "Interaction (Engrais)", col = 1:4, lwd = 2)

---
## 3. Tests non paramétriques

### 3.1 Wilcoxon vs χ² -- Round 1

In [ ]:
set.seed(42)
x = sample(1:100, 100, replace = TRUE)
Q1 = quantile(x)[2]
Q3 = quantile(x)[4]

X = x[x < Q1 | x > Q3]
Y = x[x > Q1 & x < Q3]
X = c(X, sample(Q1:Q3, 5))
Y = c(Y, sample(1:Q1, 5), sample(Q3:100, 5))
X = sort(X)
Y = sort(Y)

In [ ]:
# Test χ²
table_chi = rbind(c(1, 1, 1), c(1, 1, 1))
table_chi[1, 1] = length(X[X < Q1])
table_chi[1, 2] = length(X[X > Q1 & X < Q3])
table_chi[1, 3] = length(X[X > Q3])
table_chi[2, 1] = length(Y[Y < Q1])
table_chi[2, 2] = length(Y[Y > Q1 & Y < Q3])
table_chi[2, 3] = length(Y[Y > Q3])
chisq.test(table_chi)

In [ ]:
# Test de Wilcoxon
matX = rbind(X, rep(1, length(X)))
matY = rbind(Y, rep(0, length(Y)))
matXY = cbind(matX, matY)
mat = matXY[, order(matXY[1, ])]
data_r = rbind(mat, 1:length(mat[1, ]))
rang = data_r[3, ]
W = sum(rang[data_r[2, ] == 1])

U = W - length(X) * (length(X) + length(Y) + 1) / 2
pval_w = 1 - pnorm(abs(U), 0, sqrt(length(X) * length(Y) * (length(X) + length(Y) + 1) / 12))
cat("p-value Wilcoxon =", pval_w)

χ² détecte, Wilcoxon non → **χ² 1 - 0 Wilcoxon**.

### 3.2 Round 2

In [ ]:
X2 = x[c(1:12, 25:36, 49:60, 73:84)]
Y2 = x[c(13:24, 37:48, 61:72, 85:96)]

# Wilcoxon round 2
matX2 = rbind(X2, rep(1, length(X2)))
matY2 = rbind(Y2, rep(0, length(Y2)))
matXY2 = cbind(matX2, matY2)
mat2 = matXY2[, order(matXY2[1, ])]
data2 = rbind(mat2, 1:length(mat2[1, ]))
rang2 = data2[3, ]
W2 = sum(rang2[data2[2, ] == 1])
U2 = W2 - length(X2) * (length(X2) + length(Y2) + 1) / 2
pval2 = 1 - pnorm(abs(U2), 0, sqrt(length(X2) * length(Y2) * (length(X2) + length(Y2) + 1) / 12))
cat("p-value Wilcoxon round 2 =", pval2)

Wilcoxon détecte cette fois → **1 partout !**

### 3.3 ANOVA vs non paramétrique (Ozone)

In [ ]:
ozone = read.table("ozone.txt", sep = " ", dec = ".", header = TRUE)
head(ozone)
vent = ozone$vent
maxO3 = ozone$maxO3

In [ ]:
# ANOVA
anova(lm(maxO3 ~ vent))

In [ ]:
# Normalité par groupe
tapply(maxO3, vent, shapiro.test)

Certains groupes **non normaux** → ANOVA risquée !

In [ ]:
# Kruskal-Wallis (alternative non paramétrique)
kruskal.test(maxO3 ~ vent)

### 3.4 Corrélation de Spearman (Vx9 vs T9)

In [ ]:
Vx9 = ozone$Vx9
T9 = ozone$T9

# À la main
Rstar = cor(rank(Vx9), rank(T9))
seuil = qnorm(0.975, 0, sqrt(1 / (length(T9) - 1)))
cat("R* =", round(Rstar, 4), "  Seuil =", round(seuil, 4))
cat("\nRejet ?", abs(Rstar) > seuil)

In [ ]:
# Avec la fonction R
cor.test(Vx9, T9, method = "spearman")

**Conclusion :** Les variables sont **dépendantes**.

---
## Résumé

| Test | Normalité requise | Fonction R |
|------|:-----------------:|------------|
| ANOVA (Fisher) | Oui | `anova(lm())` |
| Kruskal-Wallis | Non | `kruskal.test()` |
| Wilcoxon | Non | Calcul manuel |
| Spearman | Non | `cor.test(method="spearman")` |